In [1]:
import polars as pl

year = 2022
poll_data = pl.scan_parquet(
    f"s3://arthurmanceau/election_modeling_uhcp/data/polls/presidentiel/{year}/polls_t1.parquet",
    storage_options={
        "aws_endpoint_url": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
    },
    credential_provider=pl.CredentialProviderAWS(
        profile_name="default",
        region_name="us-east-1",
    ),
).collect()

results = pl.scan_parquet(
    f"s3://arthurmanceau/election_modeling_uhcp/data/output/results/results_synth_{year}_pres_['TD', 'TG', 'par']_0.5.0.parquet",
    storage_options={
        "aws_endpoint_url": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
    },
    credential_provider=pl.CredentialProviderAWS(
        profile_name="default",
        region_name="us-east-1",
    ),
    glob=False,
).collect()

In [2]:
td_true = (
    results.filter(pl.col("index") == "pvoteTD").get_column(f"{year}_pres_true").item(0)
)
tg_true = (
    results.filter(pl.col("index") == "pvoteTG").get_column(f"{year}_pres_true").item(0)
)

In [ ]:
poll_data.with_columns(delta=100 - (pl.col("TG") + pl.col("TD"))).with_columns(
    tg_adjusted=pl.col("TG") + pl.col("delta") / 2,
    td_adjusted=pl.col("TD") + pl.col("delta") / 2,
).with_columns(
    mae_poll_tg=pl.col("tg_adjusted") - tg_true,
    mae_poll_td=pl.col("td_adjusted") - td_true,
).filter(pl.col("Sondeur") != "Résultats")

mae_poll_td
f64
4.33
4.58
5.83
5.08
4.83
…
1.58
0.83
0.58
